# MPI SpMV实验

## 3.1 实验概览

按顺序运行下面的代码，需要修改的参数已在代码中标出。


In [ ]:
%%bash
set -e
cmake --version
mpicxx --version
mpirun --version


<a id="s-03-02_mpi_programming_model"></a>


## 3.2 MPI 编程模型

### 本节学习目标

- 掌握 MPI 生命周期
- 比较 MPI 与 OpenMP

### MPI 生命周期

`MPI_Init` 建立运行时，`MPI_Comm_rank/size` 获取进程身份与规模，所有正常路径最终调用 `MPI_Finalize`。`MPI_COMM_WORLD` 包含本次作业启动的全部进程。

### 与 OpenMP 的差异

OpenMP 线程共享数组；MPI 的每个进程保存自己的数组。rank 是进程编号，rank 0 负责最初的数据准备和最终汇总。

| 调用 | 本实验传递什么 | 谁收到 |
|---|---|---|
| Bcast | 完整向量 x、公共参数 | 所有进程 |
| Scatterv | 不同大小的 CSR 行块 | 各进程自己的部分 |
| Gatherv | 各进程的局部 y | rank 0 |
| Reduce | 时间、误差等统计值 | rank 0 |

`counts` 是各段的元素数量，`displacements` 是各段在原数组中的起始下标。


### 预期现象与结果分析

每个 rank 都执行同一主程序，通过 rank 分支承担不同职责。发生错误时工程调用 MPI_Abort，避免其他 rank 永久等待。

### 课后实践

画出 rank 0 与其他 rank 的数据职责，并说明为何所有进程必须按同一顺序进入 collective。

参考答案见 `answer/03.02_answer.md`。

<a id="s-03-03_csr_data_partitioning"></a>


## 3.3 CSR 数据分区

### 本节学习目标

- 理解 row 与 nnz 两种平衡标准
- 追踪 Scatterv 参数

### 按非零元数量分区

工程按 `row_ptr` 的累计非零元数量寻找行边界，尽量让各进程承担相近的计算量。单个特别长的行仍可能造成负载不均，因此还需查看各进程的非零元数量和计算时间。

### 本地 CSR

rank 0 计算 row/nnz counts 和 displacements，Scatterv 分发三组数组。收到 row_ptr 后减去首个全局偏移，使本地 row_ptr 从 0 开始。

例如 `row_ptr=[0,2,8,9,12]`，两进程分别取得行 `[0,2)` 和 `[2,4)`：

| 数组 | counts | displacements |
|---|---|---|
| row_ptr | [3,3] | [0,2] |
| col_idx、values | [8,4] | [0,8] |
| 汇集 y | [2,2] | [0,2] |

row_ptr 各多传一个末尾边界，所以两段共享原数组中的下标 2。第二个进程收到 `[8,9,12]`，减去首值 8，得到本地 `[0,1,4]`。分到 0 行时，row_ptr 仍有一个边界元素，进程仍参加通信。


### 读取负载指标

`nnz_balance_ratio=max_local_nnz/average_nnz`，理想值为 1。

`load_imbalance=(max_compute-average_compute)/average_compute`，理想值为 0。先对重复轮次求出每个 rank 的平均 compute，再取这些平均时间的最大值和平均值。可用 `.per_rank.csv` 的 `compute_ms` 列计算。

汇总行的 `compute` 则先逐轮跨 rank 取最大值，再求平均，统计顺序不同，不能直接代入上面的公式。


### 课后实践

四行的非零元数量依次为 `[2, 6, 1, 3]`，全局 `row_ptr=[0,2,8,9,12]`。将连续的完整行分给两个进程，使两个进程的非零元数量尽量接近。写出行区间、各自的非零元数量和从 0 开始的本地 row_ptr，并解释为什么要减去分区的起始偏移。

参考答案见 `answer/03.03_answer.md`。

<a id="s-03-04_distributed_spmv_implementation"></a>


## 3.4 分布式 SpMV 实现

### 本节学习目标

- 串联 MPI_Bcast、local SpMV 和 MPI_Gatherv
- 理解最大时间归约


### 每轮数据流

rank 0 持有完整 x；`MPI_Bcast` 广播后，每个 rank 对本地 CSR 行执行同一串行内核；`MPI_Gatherv` 根据 row_counts/displacements 恢复全局 y。

### 时间字段包含哪些工作

CSR 分发只做一次，记为 `distribution_once`。每轮先 Barrier，再计时 Broadcast → local SpMV → Gatherv；本轮 total 不含起始 Barrier 和一次性分发。

每轮测完后跨 rank 取 MAX，最后对重复轮次求平均。这表示每轮最慢 rank 的平均耗时。各阶段的最慢 rank 可能不同，所以阶段 MAX 的和不必等于 total。

### 查看核心循环

打开 `src/mpi_spmv/src/spmv_mpi.cpp`，找到 `MPI_Bcast`、本地 SpMV 和 `MPI_Gatherv`，对照上面的数据流阅读。

### 编译并用两个进程运行

下面运行 U1 矩阵，查看各进程的分区、结果误差和耗时。`correctness_error_l2` 是绝对 L2 误差，即 `sqrt(sum((y-ref)^2))`，小于 `1e-6` 时通过检查；`correctness_error_max_abs` 是最大的逐元素绝对差。

OpenMPI 4.0.3 在 GCC 9 下可能输出 `mpicxx.h` 的 `-Wcast-function-type` 告警；若编译继续完成，可接着运行程序。


In [ ]:
%%bash
set -euo pipefail
cd src/mpi_spmv
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DCMAKE_CXX_COMPILER=mpicxx
cmake --build build -j
mkdir -p results
# 学生修改：-np 后的数字是本次启动的进程数。
mpirun -np 2 ./build/spmv_mpi --matrix U1 --warmup 1 --repeat 3 2>&1 | tee results/mpi_run.log


### 预期现象与结果分析

rank 0 应输出 CPU baseline、MPI 端到端时间、分阶段耗时、负载均衡和 PASS/FAIL。进程数不能超过调度器分配资源。

### 课后实践

解释为何分阶段最大值不能简单相加还原端到端最大值。

参考答案见 `answer/03.04_answer.md`。

<a id="s-03-05_mpi_scaling_analysis"></a>


## 3.5 MPI 扩展性分析

### 本节学习目标

- 比较不同进程数
- 识别通信主导区

### 扩展性

进程增加会缩短局部行块，但完整 x 仍需广播、y 仍需汇聚。小矩阵固定通信开销占比更高，大矩阵也可能受到内存带宽和网络限制。

### 选择进程数

进程数不超过分配的 CPU 核心数，避免多个进程争用同一核心影响比较。各次运行使用相同矩阵、预热和重复次数。

### 预期现象与结果分析

扩展性报告应同时包含端到端、通信、计算和负载均衡。即使 local compute 下降，总时间也可能因通信增长而上升。

### 课后实践

设计不超过可用 CPU slot 的进程数实验，并说明选择依据。

参考答案见 `answer/03.05_answer.md`。